# Test detector YOLO

In [ ]:
from ultralytics import YOLO
import torch

torch.cuda.empty_cache()  
torch.backends.cudnn.benchmark = True  
torch.backends.cudnn.enabled = True  

# Modificare i percorsi secondo necessità
# Il percorso dei pesi del modello da testare
MODEL_WEIGHTS = "./detection models/ft_1920_yolo11m.pt"

DATASET_YAML = "./working/data_sampled/data.yaml"  

DEVICE = "cuda:0"
IMG_SIZE = 1920
CONF_THRES = 0.001  
IOU_THRES = 0.60    

def main():
    print(f"Caricamento Modello: {MODEL_WEIGHTS}")
    model = YOLO(MODEL_WEIGHTS)

    print(f"Avvio Valutazione su: {DATASET_YAML}")
    
    metrics = model.val(
        data=DATASET_YAML,
        imgsz=IMG_SIZE,
        batch=8,         
        conf=CONF_THRES,  
        iou=IOU_THRES,    
        device=DEVICE,
        split='val',     
        half=True,        
        workers=8        
    )

    print("RISULTATI:")
    print(f"Precision (P):    {metrics.box.mp:.4f}")
    print(f"Recall (R):       {metrics.box.mr:.4f}")
    print(f"mAP@50:           {metrics.box.map50:.4f}")
    print(f"mAP@50-95:        {metrics.box.map:.4f}") 

if __name__ == "__main__":
    main()

# Test con field detection

In [ ]:
import os
import cv2
import yaml
import glob
import torch
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from torchvision.ops import box_iou

# Modificare i percorsi secondo necessità
MODEL_WEIGHTS = "./detection models/others/yolo11m.pt"
DATASET_YAML = "./working/data_sampled/data.yaml"
DEVICE = "cuda:0"
IMG_SIZE = 1920

CONF_THRES = 0.001  
IOU_THRES  = 0.6    

# SELETTORE MODALITÀ 
# "static"   -> Usa range HSV fissi 
# "adaptive" -> Calibra il colore dell'erba per ogni immagine 
DETECTOR_MODE = "adaptive"  

class StaticFieldDetector:
    """
    Rilevatore di campo con soglie HSV fisse.
    """
    def __init__(self):
        # Range verde standard
        self.lower_green = np.array([35, 40, 40])
        self.upper_green = np.array([90, 255, 255])
        self.kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

    def get_field_mask(self, frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, self.lower_green, self.upper_green)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, self.kernel)
        
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return np.zeros_like(mask)
            
        largest_contour = max(contours, key=cv2.contourArea)
        hull = cv2.convexHull(largest_contour)
        
        field_mask = np.zeros_like(mask)
        cv2.drawContours(field_mask, [hull], -1, 255, thickness=cv2.FILLED)
        return field_mask


class AdaptiveFieldDetector:
    """
    Rilevatore adattivo: analizza la parte bassa dell'immagine per capire
    il tono specifico di verde del campo corrente.
    """
    def __init__(self):
        self.lower_green = None
        self.upper_green = None
        self.kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
        self.calibrated = False

    def calibrate(self, frame):
        """Trova il colore dominante nel terzo inferiore centrale."""
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        h, w, _ = hsv.shape
        
        roi = hsv[int(h*0.6):int(h*0.95), int(w*0.2):int(w*0.8)]

        # Istogramma Hue
        hist_h = cv2.calcHist([roi], [0], None, [180], [0, 180])
        dominant_hue = np.argmax(hist_h)

        # Definizione range dinamico intorno al picco
        if 25 < dominant_hue < 95:
            self.lower_green = np.array([max(0, dominant_hue - 15), 40, 40])
            self.upper_green = np.array([min(180, dominant_hue + 15), 255, 255])
        else:
            # Fallback se il colore rilevato non è verde
            self.lower_green = np.array([35, 40, 40])
            self.upper_green = np.array([85, 255, 255])

        self.calibrated = True

    def get_field_mask(self, frame):
        if not self.calibrated:
            self.calibrate(frame)

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, self.lower_green, self.upper_green)

        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, self.kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, self.kernel)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return np.zeros_like(mask)

        largest_contour = max(contours, key=cv2.contourArea)
        
        epsilon = 0.01 * cv2.arcLength(largest_contour, True)
        approx_curve = cv2.approxPolyDP(largest_contour, epsilon, True)
        hull = cv2.convexHull(approx_curve)

        field_mask = np.zeros_like(mask)
        cv2.drawContours(field_mask, [hull], -1, 255, thickness=cv2.FILLED)
        return field_mask

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.], recall, [1.]))
    mpre = np.concatenate(([0.], precision, [0.]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

def get_batch_statistics(outputs, targets, iou_threshold):
    if len(targets) == 0: return np.zeros(len(outputs)), np.zeros(len(outputs)), 0
    if len(outputs) == 0: return np.array([]), np.array([]), len(targets)
    
    target_boxes = torch.tensor(targets)
    output_boxes = torch.tensor(outputs)
    
    iou_matrix = box_iou(output_boxes, target_boxes).numpy()
    tp = np.zeros(len(outputs))
    detected_gt = []
    
    for i, _ in enumerate(outputs):
        if iou_matrix.shape[1] == 0: break
        gt_idx = np.argmax(iou_matrix[i])
        max_iou = iou_matrix[i, gt_idx]
        if max_iou >= iou_threshold and gt_idx not in detected_gt:
            tp[i] = 1
            detected_gt.append(gt_idx)
            
    return tp, np.ones(len(outputs)) - tp, len(targets)

def load_dataset_images(yaml_path, split='val'):
    print(f"Reading dataset: {yaml_path}")
    with open(yaml_path, 'r') as f: data_cfg = yaml.safe_load(f)
    path_root = data_cfg.get('path', '')
    img_dir = data_cfg.get(split)
    if not img_dir: raise ValueError(f"Key '{split}' not found in YAML.")
    
    full_path = os.path.join(path_root, img_dir) if not os.path.isabs(img_dir) else img_dir
    images = []
    if os.path.isdir(full_path):
        for ext in ['*.jpg']:
            images.extend(glob.glob(os.path.join(full_path, ext)))
    elif os.path.isfile(full_path):
         with open(full_path, 'r') as f:
            lines = f.read().splitlines()
            images = [os.path.join(path_root, l) if not os.path.isabs(l) else l for l in lines]
    return sorted(images)

def main():
    torch.cuda.empty_cache()
    print(f"Loading Model: {MODEL_WEIGHTS}")
    model = YOLO(MODEL_WEIGHTS)
    
    images = load_dataset_images(DATASET_YAML, split='val')
    print(f"Found {len(images)} validation images.")
    print(f"Active Detector Mode: {DETECTOR_MODE.upper()}")

    static_detector = StaticFieldDetector() if DETECTOR_MODE == "static" else None
    
    stats = []
    
    print("Starting Validation Loop...")
    for img_path in tqdm(images):
        frame = cv2.imread(img_path)
        if frame is None: continue
        h, w = frame.shape[:2]
        
        # Field mask generation
        if DETECTOR_MODE == "adaptive":
            field_detector = AdaptiveFieldDetector()
            field_mask = field_detector.get_field_mask(frame)
        else:
            field_mask = static_detector.get_field_mask(frame)

        # label loading
        label_path = img_path.replace('images', 'labels').replace('img1', 'labels_with_ids').rsplit('.', 1)[0] + ".txt"
        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if int(parts[0]) == 0: # Solo classe 0 
                        cx, cy, bw, bh = parts[1:]
                        x1, y1 = (cx - bw/2) * w, (cy - bh/2) * h
                        x2, y2 = (cx + bw/2) * w, (cy + bh/2) * h
                        gt_boxes.append([x1, y1, x2, y2])
        
        # Inferenza
        results = model(frame, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, classes=[0], verbose=False)[0]
        
        if results.boxes is None:
             stats.append({'boxes': np.array([]), 'conf': np.array([]), 'gt': np.array(gt_boxes)})
             continue

        pred_boxes = results.boxes.xyxy.cpu().numpy()
        pred_conf = results.boxes.conf.cpu().numpy()

        # Filtraggio con maschera campo
        valid_boxes, valid_confs = [], []
        
        for box, conf in zip(pred_boxes, pred_conf):
            x1, y1, x2, y2 = map(int, box)
            # Calcolo posizione piedi 
            feet_x = max(0, min(int((x1+x2)/2), w-1))
            feet_y = max(0, min(int(y2), h-1))
            
            if field_mask[feet_y, feet_x] > 0:
                valid_boxes.append([x1, y1, x2, y2])
                valid_confs.append(conf)
        
        stats.append({ 
            'boxes': np.array(valid_boxes), 
            'conf': np.array(valid_confs), 
            'gt': np.array(gt_boxes) 
        })

    print("\nComputing Metrics...")
    
    iou_thresholds = np.linspace(0.5, 0.95, 10)
    ap_per_iou = []
    
    final_p, final_r, final_f1 = 0.0, 0.0, 0.0

    for i, iou_thresh in enumerate(iou_thresholds):
        tp_list, conf_list = [], []
        num_gt_total = 0
        
        for item in stats:
            p_boxes, p_conf, gt = item['boxes'], item['conf'], item['gt']
            if len(p_conf) > 0:
                sort_idx = np.argsort(-p_conf)
                p_boxes, p_conf = p_boxes[sort_idx], p_conf[sort_idx]
            
            tp, fp, n_gt = get_batch_statistics(p_boxes, gt, iou_thresh)
            tp_list.append(tp)
            conf_list.append(p_conf)
            num_gt_total += n_gt

        if len(tp_list) > 0 and num_gt_total > 0:
            tp_all = np.concatenate(tp_list)
            conf_all = np.concatenate(conf_list)
            
            sort_idx = np.argsort(-conf_all)
            tp_all = tp_all[sort_idx]
            
            tp_cumsum = np.cumsum(tp_all)
            fp_cumsum = np.cumsum(1 - tp_all)
            
            recall = tp_cumsum / (num_gt_total + 1e-16)
            precision = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-16)
            
            ap = compute_ap(recall, precision)
            ap_per_iou.append(ap)

            if i == 0: 
                f1_scores = 2 * (precision * recall) / (precision + recall + 1e-16)
                best_i = np.argmax(f1_scores)
                final_p = precision[best_i]
                final_r = recall[best_i]
                final_f1 = f1_scores[best_i]
        else:
            ap_per_iou.append(0.0)

    map50 = ap_per_iou[0]
    map5095 = np.mean(ap_per_iou)

    print(f"RISULTATI (Mode: {DETECTOR_MODE})")
    print(f"Precision:   {final_p:.4f}")
    print(f"Recall:      {final_r:.4f}")
    print(f"F1 Score:    {final_f1:.4f}")
    print(f"mAP@50:      {map50:.4f}")
    print(f"mAP@50-95:   {map5095:.4f}")

if __name__ == "__main__":
    main()